# 04 Graph



Build the room graph from `Rhino_Geometry&Apertures.obj`, keeping the same room naming convention as the earlier primal/dual notebook.



The workflow is:

1. Load the combined Rhino OBJ in its exported direction.

2. Build the room `CellComplex` and inspect the primal and dual graphs.

3. Add the imported windows and name them by the room they belong to.


In [50]:
from pathlib import Path
import colorsys
from collections import Counter



from topologicpy.Vertex import Vertex
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper



print(Helper.Version())

renderer = 'vscode'

ROOT = Path.cwd()

OBJ_PATH = ROOT / 'Rhino_Geometry&Apertures.obj'


The version that you are using (0.9.18) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 1. Import the Combined OBJ



The combined Rhino OBJ is loaded in the same direction it was exported. The room topology below follows that same imported orientation, so no extra axis remap is applied.


In [51]:
obj_parts = Topology.ByOBJPath(str(OBJ_PATH))

obj_parts = sorted(

    obj_parts,

    key=lambda part: len(Topology.Faces(Cluster.ByTopologies([part])) or [])

)

aperture_cluster, shell_cluster = obj_parts

apertures = Topology.Faces(Cluster.ByTopologies([aperture_cluster])) or []

shell_faces = Topology.Faces(Cluster.ByTopologies([shell_cluster])) or []



print(f'Loaded {len(obj_parts)} topologies from {OBJ_PATH.name}')

print(f'Shell faces: {len(shell_faces)}')

print(f'Window faces: {len(apertures)}')



Topology.Show(

    shell_cluster, aperture_cluster,

    faceColor=[180, 205, 245], faceOpacity=0.18,

    edgeColor='white', edgeWidth=0.5,

    vertexColor='red', vertexSize=4,

    showVertices=False,

    backgroundColor='black',

    width=900, height=650,

    renderer=renderer

)


Topology.ByOBJPath - Error: the input OBJ path does not exist. Returning None.


TypeError: 'NoneType' object is not iterable

## 2. Rebuild the Rooms

The OBJ is still face-based, so the rooms are reconstructed as cells. The room names follow the same convention as before: Bedroom 1, Corridor, Bedroom 2, WC 1, WC 2, Kitchen, Living Room, WC, Office, Pantry.

In [ ]:
def make_room(xy_polygon, z0=0, z1=10):

    n = len(xy_polygon)

    bot = [Vertex.ByCoordinates(x, y, z0) for x, y in xy_polygon]

    top = [Vertex.ByCoordinates(x, y, z1) for x, y in xy_polygon]

    faces = [Face.ByVertices(bot), Face.ByVertices(list(reversed(top)))]

    for i in range(n):

        j = (i + 1) % n

        faces.append(Face.ByVertices([bot[i], top[i], top[j], bot[j]]))

    return Cell.ByFaces(faces)





def box(x0, y0, x1, y1, z0=0, z1=10):

    xmn, xmx = min(x0, x1), max(x0, x1)

    ymn, ymx = min(y0, y1), max(y0, y1)

    return make_room([(xmn, ymn), (xmx, ymn), (xmx, ymx), (xmn, ymx)], z0, z1)





def centroid_xyz(topology):

    c = Topology.Centroid(topology)

    return (c.X(), c.Y(), c.Z())





def dist3(a, b):

    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5





def face_profile(face):

    verts = Topology.Vertices(face) or []

    xs = [v.X() for v in verts]

    ys = [v.Y() for v in verts]

    zs = [v.Z() for v in verts]

    return {

        'face': face,

        'centroid': centroid_xyz(face),

        'normal': Face.Normal(face),

        'xmin': min(xs), 'xmax': max(xs),

        'ymin': min(ys), 'ymax': max(ys),

        'zmin': min(zs), 'zmax': max(zs),

    }





def is_vertical(face, tolerance=0.1):

    return abs(Face.Normal(face)[2]) <= tolerance





def assign_window_to_room(aperture, room_face_data, alignment_threshold=0.95, penalty_weight=10):

    aperture_data = face_profile(aperture)

    best_match = None

    best_score = float('inf')

    for room_face in room_face_data:

        alignment = abs(sum(a * b for a, b in zip(aperture_data['normal'], room_face['normal'])))

        if alignment < alignment_threshold:

            continue

        dx = max(room_face['xmin'] - aperture_data['xmin'], 0, aperture_data['xmax'] - room_face['xmax'])

        dy = max(room_face['ymin'] - aperture_data['ymin'], 0, aperture_data['ymax'] - room_face['ymax'])

        dz = max(room_face['zmin'] - aperture_data['zmin'], 0, aperture_data['zmax'] - room_face['zmax'])

        score = dist3(aperture_data['centroid'], room_face['centroid']) + penalty_weight * (dx + dy + dz)

        if score < best_score:

            best_score = score

            best_match = room_face['room']

    return best_match


In [ ]:
rooms = [
    box(75, -7, 99, 17),
    box(75, -43, 78, -7),
    make_room([(78, -7), (99, -7), (99, -22), (86, -22), (86, -19), (78, -19)]),
    box(78, -25, 86, -19),
    box(78, -31, 86, -25),
    make_room([(78, -31), (78, -43), (99, -43), (99, -22), (86, -22), (86, -31)]),
    make_room([(35, -24), (60, -24), (60, -7), (75, -7), (75, 0), (35, 0)]),
    box(5, -17, 15, -7),
    make_room([(0, -17), (0, 0), (35, 0), (35, -24), (15, -24), (15, -7), (5, -7), (5, -17)]),
    box(0, -24, 15, -17),
]

room_reference_points = [
    (87.0, 5.0, 5.0),
    (76.5, -25.0, 5.0),
    (89.0, -14.0, 5.0),
    (82.0, -22.0, 5.0),
    (82.0, -28.0, 5.0),
    (89.8, -33.7, 5.0),
    (50.5, -10.7, 5.0),
    (10.0, -12.0, 5.0),
    (20.3, -10.6, 5.0),
    (7.5, -20.5, 5.0),
]

room_names = [
    'Bedroom 1', 'Corridor', 'Bedroom 2', 'WC 1', 'WC 2',
    'Kitchen', 'Living Room', 'WC', 'Office', 'Pantry'
]

cc = CellComplex.ByCells(rooms)
cells = Topology.Cells(cc)
colors = [
    '#%02x%02x%02x' % tuple(int(c * 255) for c in colorsys.hsv_to_rgb(i / len(cells), 0.55, 0.9))
    for i in range(len(cells))
]

room_data = []
for i, cell in enumerate(cells):
    cp = centroid_xyz(cell)
    idx = min(range(len(room_reference_points)), key=lambda j: dist3(cp, room_reference_points[j]))
    room_name = room_names[idx]
    Topology.SetDictionary(
        cell,
        Dictionary.ByKeysValues(['id', 'name', 'color'], [i, room_name, colors[i]])
    )
    room_data.append({'cell': cell, 'name': room_name, 'centroid': cp})
    print(f'cell {i}: {room_name} @ ({cp[0]:.1f}, {cp[1]:.1f}, {cp[2]:.1f})')

Topology.Show(
    cc,
    faceColorKey='color', faceOpacity=0.45,
    edgeColor='white', edgeWidth=1,
    showVertices=False,
    backgroundColor='black',
    width=900, height=650,
    renderer=renderer
)

cell 0: Bedroom 1 @ (83.5, 2.2, 5.0)
cell 1: Corridor @ (77.1, -25.0, 5.0)
cell 2: Bedroom 2 @ (87.7, -16.0, 5.0)
cell 3: Living Room @ (56.7, -10.3, 5.0)
cell 4: Kitchen @ (87.4, -31.0, 5.0)
cell 5: WC 2 @ (82.0, -28.0, 5.0)
cell 6: WC 1 @ (82.8, -22.0, 5.0)
cell 7: WC @ (13.9, -12.6, 5.0)
cell 8: Pantry @ (7.0, -19.8, 5.0)
cell 9: WC @ (10.0, -12.0, 5.0)


## 3. Primal and Dual

The primal graph is the room topology itself: corners and edges of the room cell complex. The dual graph collapses each room into one node and connects rooms that share a boundary.

In [ ]:
print(f'Primal graph: {len(Topology.Vertices(cc))} vertices, {len(Topology.Edges(cc))} edges')

Topology.Show(
    cc,
    faceColor='steelblue', faceOpacity=0.15,
    edgeColor='red', edgeWidth=2,
    vertexColor='red', vertexSize=8,
    showVertices=True,
    backgroundColor='black',
    width=900, height=650,
    renderer=renderer
)

g_dual = Graph.ByTopology(cc)
print(f'Dual graph: {len(Graph.Vertices(g_dual))} nodes, {len(Graph.Edges(g_dual))} edges')

for vertex, room in zip(Graph.Vertices(g_dual), room_data):
    Topology.SetDictionary(
        vertex,
        Dictionary.ByKeysValues(['label', 'size', 'color'], [room['name'], 16, 'red'])
    )
for edge in Graph.Edges(g_dual):
    Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width', 'color'], [4, 'white']))

Topology.Show(
    cc, g_dual,
    faceColor='blue', faceOpacity=0.15,
    edgeColor='white', edgeWidth=0.5,
    vertexColorKey='color', vertexSizeKey='size',
    edgeColorKey='color', edgeWidthKey='width',
    showVertices=True,
    backgroundColor='black',
    width=900, height=650,
    renderer=renderer
)

Primal graph: 58 vertices, 105 edges


Dual graph: 10 nodes, 16 edges


## 4. Add the Windows



The smaller imported cluster contains the window faces. To keep every window exactly where Rhino exported it, the notebook leaves that geometry untouched, matches each window to the closest vertical room face, and visualizes only the exported windows in this step.


In [ ]:
window_counts = Counter()

window_labels = []

room_face_data = []



for room in room_data:

    for face in Topology.Faces(room['cell']) or []:

        if is_vertical(face):

            room_face = face_profile(face)

            room_face['room'] = room['name']

            room_face_data.append(room_face)



for aperture in apertures:

    room_name = assign_window_to_room(aperture, room_face_data) or 'Unassigned'

    window_counts[room_name] += 1

    window_name = f'{room_name} Window {window_counts[room_name]}'

    Topology.SetDictionary(

        aperture,

        Dictionary.ByKeysValues(['name', 'room', 'color'], [window_name, room_name, '#ffc857'])

    )

    window_labels.append(window_name)



window_cluster = Cluster.ByTopologies(apertures)



print(f'Kept {len(apertures)} exported windows in place')

for room_name in sorted(window_counts):

    print(f'{room_name}: {window_counts[room_name]} window(s)')



Topology.Show(

    window_cluster,

    faceColorKey='color', faceOpacity=0.95,

    edgeColor='white', edgeWidth=1,

    showVertices=False,

    backgroundColor='black',

    width=900, height=650,

    renderer=renderer

)


Kept 33 exported windows in place
Bedroom 1: 16 window(s)
Bedroom 2: 1 window(s)
Living Room: 8 window(s)
WC: 8 window(s)


## 5. Optional Context View



This view keeps the exported windows in place and overlays them on a very faint shell, so you can verify location without the shell dominating the plot.


In [ ]:
Topology.Show(

    shell_cluster, window_cluster,

    faceColor=[170, 190, 220], faceOpacity=0.04,

    edgeColor='gray', edgeWidth=0.25,

    showVertices=False,

    backgroundColor='black',

    width=900, height=650,

    renderer=renderer

)


In [ ]:
window_labels[:10]

['Bedroom 1 Window 1',
 'WC Window 1',
 'WC Window 2',
 'WC Window 3',
 'WC Window 4',
 'WC Window 5',
 'Living Room Window 1',
 'Living Room Window 2',
 'WC Window 6',
 'WC Window 7']